# Text matching / retrieval · toolkit modular v2

Queries perlu relevant_ids JSON list. Benchmark proxy ringkasan-artikel terlalu mudah; gunakan judgment kueri nyata untuk keputusan final.

Ubah konfigurasi. Hapus/tambah blok melalui perakit, bukan mengubah puluhan sel. Semua model menggunakan split/urutan label yang sama. Hasil harus dihitung ulang di Kaggle; skor notebook sumber tidak diwariskan.

Sumber adaptasi: `notebook/e_pasangan_teks/22_retrieval/22_retrieval_review.ipynb`.

## 1 · Konfigurasi

File, kolom, preprocessing dan model hanya di sini. Group ID pengguna opsional. Untuk offline, set offline=True dan ganti model path ke bundle Kaggle input.

In [ ]:
CFG = {'encoding': 'utf-8',
 'seed': 42,
 'out': '/kaggle/working/hasil_toolkit',
 'offline': False,
 'preprocess': 'native',
 'slang': {},
 'stopwords': [],
 'eda_sample': 5000,
 'eda_top': 15,
 'eda_languages': False,
 'max_features': 60000,
 'max_len': 192,
 'e5_model': 'intfloat/multilingual-e5-base',
 'corpus_file': '/kaggle/input/GANTI_DATASET/corpus.csv',
 'queries_file': '/kaggle/input/GANTI_DATASET/queries.csv',
 'doc_id_col': 'doc_id',
 'doc_text_col': 'text',
 'query_col': 'query',
 'relevant_col': 'relevant_ids',
 'task': 'retrieval',
 'models': ['tfidf', 'bm25', 'e5', 'rrf', 'reranker'],
 'sentiment_pretrained': False}

## 2 · Lingkungan Kaggle

Pasang hanya dependensi komponen terpilih. Untuk bundle offline: rakit dengan --no-install atau hapus sel instalasi.

In [ ]:
import os, sys, subprocess
from pathlib import Path
assert Path('/kaggle/input').exists(), 'Jalankan di Kaggle.'
os.environ.update(USE_TF='0', USE_FLAX='0', TOKENIZERS_PARALLELISM='false')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'accelerate>=1,<2', 'emoji>=2,<3', 'ftfy>=6.3,<7', 'numpy', 'pandas', 'rank-bm25==0.2.2', 'scikit-learn', 'scipy', 'sentencepiece>=0.2,<1', 'torch', 'transformers==5.20.2'], check=True)

## 3 · Fungsi komponen

Definisi sekali; sel pemakaian pendek berada di bawah. Semua dependensi disertakan, tanpa import dari folder toolkit.

### Kontrak, data loading & penyimpanan



In [ ]:
"""Kontrak bersama. Tidak memasang paket atau menjalankan model saat disalin."""
import json
import random
import re
import time
from pathlib import Path
import numpy as np
import pandas as pd

MODEL_RUNNERS = {}
ENCODERS = {}


def encode_e5(texts, cfg, role='query'):
    assert 'e5' in ENCODERS, 'Pasang komponen encoder E5 melalui perakit.'
    return ENCODERS['e5'](texts,cfg,role)


def register(name):
    def decorate(fn):
        MODEL_RUNNERS[name] = fn
        return fn
    return decorate


def read_table(path, encoding="utf-8", string_cols=()):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Isi CFG['data_file'] dengan file Kaggle yang ada: {path}")
    if path.suffix == ".parquet":
        return pd.read_parquet(path)
    if path.suffix in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    return pd.read_csv(path, sep="\t" if path.suffix == ".tsv" else ",", encoding=encoding,
        dtype={col:str for col in string_cols} or None)


def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)


def target_array(frame, cfg):
    return frame[cfg['label_cols']].to_numpy(float) if cfg['task']=='multilabel' else frame.y.to_numpy()


def require_kaggle(gpu=False):
    assert Path('/kaggle/input').exists(), "Jalankan eksperimen di Kaggle."
    if gpu:
        import torch
        assert torch.cuda.is_available(), "Aktifkan GPU Kaggle."


def show_table(name, table):
    from IPython.display import display
    print("\n" + name)
    display(table.to_frame() if isinstance(table, pd.Series) else table)


def save_tables(tables, out, prefix=""):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_csv(out / f"{prefix}{name}.csv", index=True)


### Preprocessing native / normalized / BoW / IndoBERTweet

Slang hanya dari CFG["slang"]. Native mempertahankan negasi, emoji, angka, case. Uji normalisasi sebagai eksperimen.

In [ ]:
import html
import unicodedata

NEGATIONS = {'tidak', 'tak', 'bukan', 'belum', 'jangan', 'tanpa', 'gak', 'nggak', 'ga', 'not', 'never'}


def safe_text(value):
    """Missing tetap menjadi kosong; angka, emoji, negasi, tanda baca dipertahankan."""
    if not isinstance(value, str):
        return ''
    import ftfy
    text = ftfy.fix_text(value, unescape_html=False, uncurl_quotes=False,
        fix_latin_ligatures=False, fix_character_width=False, remove_control_chars=False, normalization=None)
    text = unicodedata.normalize('NFC', html.unescape(text))
    text = text.translate(dict.fromkeys(map(ord, '\ufeff\u200b\u2060\x00'), None))
    text = re.sub(r'[\x01-\x08\x0b\x0c\x0e-\x1f\x7f]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def preprocess_text(value, preset='native', slang=None, stopwords=()):
    text = safe_text(value)
    if preset == 'native':
        return text
    if preset == 'indobertweet':
        import emoji
        text = text.lower()
        text = re.sub(r'https?://\S+|www\.\S+|\burl\b', 'HTTPURL', text)
        text = re.sub(r'(?<!\w)@\w+|\buser\b', '@USER', text)
        return safe_text(emoji.demojize(text))
    if preset not in {'normalized', 'bow'}:
        raise ValueError(f'Preset tidak dikenal: {preset}')
    text = text.casefold()
    if slang:
        text = re.sub(r'\b[^\W_]+\b', lambda m: slang.get(m[0], m[0]), text)
    if preset == 'bow':
        text = re.sub(r'https?://\S+|www\.\S+|@\w+', ' ', text)
        # Stopword diberikan pengguna, negasi selalu dilindungi; tidak ada stemming otomatis.
        blocked = set(stopwords) - NEGATIONS
        text = ' '.join(w for w in re.findall(r'[^\W\d_]+', text) if w not in blocked)
    return safe_text(text)


def make_cleaner(cfg):
    return lambda value: preprocess_text(value, cfg['preprocess'], cfg.get('slang'), cfg.get('stopwords', ()))


### EDA angka: missing → vocabulary → noise

Basic/noise seluruh baris; vocab/ngram memakai sampel eksplisit. Persentase noise boleh overlap. Bahasa adalah estimasi sampel.

In [ ]:
from collections import Counter

NOISE_PATTERNS = {
    'url': r'https?://|www\.', 'mention': r'@\w+', 'hashtag': r'#\w+', 'angka': r'\d',
    'emoji_simbol': r'[\U0001F300-\U0001FAFF\u2600-\u27BF]', 'huruf_ulang': r'([^\W\d_])\1{2,}',
    'baca_ulang': r'[!?.,]{2,}', 'html': r'</?[a-zA-Z][^>]*>', 'encoding': r'\ufffd|Ã.|Â.',
    'non_ascii': r'[^\x00-\x7f]', 'spasi_ulang': r' {2,}', 'invisible': r'[\u200b\ufeff\u2060]',
    'kapital': r'(?-i:\b[A-Z]{3,}\b)', 'huruf_angka': r'\b(?=\w*[a-z])(?=\w*\d)\w+\b',
    'slang_contoh': r'\b(?:gk|gak|nggak|ga|bgt|yg|utk|udh|klo|jg)\b',
    'negasi': r'\b(?:tidak|tak|bukan|belum|jangan|tanpa|gak|nggak|ga|not|never)\b'
}


def numeric_eda(frame, text_col, label_cols=(), top=15, sample=5000, seed=42, languages=False):
    """Hitungan dasar seluruh baris; vocabulary/ngram pada sampel yang ukurannya dilaporkan."""
    s = frame[text_col]
    text = s.where(s.map(lambda x: isinstance(x, str)), '').fillna('')
    nonempty = text.str.strip().ne('')
    pseudo = text.str.strip().str.casefold().isin({'nan', 'none', 'null', 'n/a', 'na', '-', '--'})
    quality = pd.Series({'rows': len(s), 'missing': s.isna().sum(), 'blank': (~nonempty).sum(),
        'pseudo_null': pseudo.sum(), 'non_string_nonmissing': (s.notna() & ~s.map(lambda x: isinstance(x, str))).sum(),
        'unique_nonempty': text[nonempty].nunique(), 'duplicate_extra_nonempty': text[nonempty].duplicated().sum(),
        'duplicate_members_nonempty': text[nonempty].duplicated(keep=False).sum()})
    subset = text[nonempty].sample(min(int(sample), int(nonempty.sum())), random_state=seed)
    tokens = subset.str.casefold().str.findall(r'[^\W\d_]+')
    counts = pd.Series(Counter(w for row in tokens for w in row), dtype='int64').sort_values(ascending=False)
    n_tokens = int(counts.sum())
    tables = {'quality': quality.to_frame('n'),
        'missing_unique': pd.DataFrame({'missing': frame.isna().sum(), 'missing_pct': frame.isna().mean()*100,
            'unique': frame.apply(lambda col:col.map(lambda value:json.dumps(value,sort_keys=True) if isinstance(value,(list,dict)) else value).nunique())}),
        'length': pd.DataFrame({'char': text.str.len(), 'word': text.str.split().str.len()}).describe(percentiles=[.01,.5,.9,.95,.99]).T,
        'vocabulary': pd.Series({'sample_rows': len(subset), 'tokens': n_tokens, 'types': len(counts),
            'hapax_types': counts.eq(1).sum(), 'hapax_pct_types': counts.eq(1).mean()*100,
            'hapax_pct_tokens': counts.eq(1).sum()/max(n_tokens,1)*100}).to_frame('value'),
        'top_words': counts.head(top).to_frame('n').assign(pct_tokens=lambda d: d.n/max(n_tokens,1)*100)}
    for n in (2,3):
        grams = Counter(' '.join(row[i:i+n]) for row in tokens for i in range(len(row)-n+1))
        tables[f'ngram_{n}'] = pd.DataFrame(grams.most_common(top), columns=['term', 'n'])
    flags = pd.DataFrame({name: text.str.count(pattern, flags=re.I).gt(0) for name,pattern in NOISE_PATTERNS.items()})
    tables['noise'] = pd.DataFrame({'n_rows': flags.sum(), 'pct_all_rows_overlap_allowed': flags.mean()*100})
    tables['odd_words_candidates'] = counts[[len(w)>=20 or bool(re.search(r'([^\W\d_])\1{2,}',w)) for w in counts.index]].head(top).to_frame('n')
    for label in label_cols:
        known = frame[label].dropna()
        count = known.value_counts()
        tables[f'label_{label}'] = count.to_frame('n').assign(pct_known=lambda d:d.n/max(len(known),1)*100)
        tables[f'conflict_{label}'] = frame.assign(_key=text.str.casefold()).loc[nonempty].groupby('_key')[label].nunique().value_counts().to_frame('text_groups')
    if languages and len(subset):
        from langid.langid import LanguageIdentifier, model
        detector = LanguageIdentifier.from_modelstring(model, norm_probs=True)
        small = subset.sample(min(150,len(subset)),random_state=seed)
        pairs = small.map(lambda t: detector.classify(t[:3000]) if len(t.split())>=5 else ('uncertain',0))
        lang = pairs.map(lambda p:p[0] if p[1]>=.8 else 'uncertain').value_counts()
        tables['language_estimate_sample_only'] = lang.to_frame('n').assign(pct_sample=lang/len(small)*100)
    return tables


def multilabel_eda(frame, labels):
    y = frame[labels].astype(float)
    complete = y.dropna()
    co = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for a in labels:
        for b in labels:
            known = y[[a,b]].notna().all(axis=1)
            co.loc[a,b] = (y.loc[known,a].eq(1)&y.loc[known,b].eq(1)).mean()*100
    return {'label_support': pd.DataFrame({'known':y.notna().sum(), 'unknown':y.isna().sum(),
                'positive':y.eq(1).sum(), 'negative':y.eq(0).sum(), 'positive_pct_known':y.mean()*100}),
        'cardinality_complete_rows': complete.sum(axis=1).value_counts().sort_index().to_frame('n'),
        'label_combinations_complete_rows': complete.value_counts().head(15).to_frame('n'),
        'cooccurrence_pct_pairwise_known': co}


def pair_eda(frame, a, b, target):
    left,right = frame[a].map(safe_text),frame[b].map(safe_text)
    pairs = pd.DataFrame({'a':left.str.casefold(),'b':right.str.casefold()})
    lset = pairs.a.map(lambda t:set(re.findall(r'[^\W\d_]+',t)))
    rset = pairs.b.map(lambda t:set(re.findall(r'[^\W\d_]+',t)))
    overlap = pd.Series([len(x&y)/max(len(x|y),1) for x,y in zip(lset,rset)])
    reversed_keys = set(zip(pairs.b,pairs.a))
    numbers = pd.Series([set(re.findall(r'\d+',x))==set(re.findall(r'\d+',y)) for x,y in zip(left,right)])
    detail = pd.DataFrame({'label_or_score':frame[target].to_numpy(),'jaccard':overlap,
        'abs_word_length_difference':(left.str.split().str.len()-right.str.split().str.len()).abs(),
        'identical':pairs.a.eq(pairs.b),'same_numbers':numbers})
    return {'pair_quality':pd.Series({'rows':len(frame),'identical_sides':pairs.a.eq(pairs.b).sum(),
            'duplicate_ordered_pairs_extra':pairs.duplicated().sum(),
            'pairs_with_reverse_present':sum((x,y) in reversed_keys for x,y in zip(pairs.a,pairs.b)),
            'shared_endpoints':len(set(pairs.a)&set(pairs.b))}).to_frame('n'),
        'pair_relation_statistics':detail.drop(columns='label_or_score').describe().T,
        'pair_relation_by_target':detail.groupby('label_or_score').agg(n=('jaccard','size'),mean_jaccard=('jaccard','mean'),
            mean_length_diff=('abs_word_length_difference','mean'),identical_pct=('identical',lambda v:v.mean()*100))}


def eda_for_task(frame, cfg):
    task = cfg['task'];tables={}
    columns = cfg['pair_cols'] if task in {'nli','similarity'} else [cfg.get('doc_text_col',cfg.get('text_col'))]
    labels = cfg['label_cols'] if task=='multilabel' else [cfg['label_col']] if task in {'classification','nli','similarity'} else []
    for i,col in enumerate(columns):
        part = numeric_eda(frame,col,labels if i==0 else (),cfg.get('eda_top',15),cfg['eda_sample'],cfg['seed'],cfg['eda_languages'])
        tables.update({str(col)+'_'+name:table for name,table in part.items()})
    if task in {'nli','similarity'}:
        tables.update(pair_eda(frame,*cfg['pair_cols'],cfg['label_col']))
    for name,table in tables.items():
        show_table(name,table)
    save_tables(tables,cfg['out'],'raw_eda_')
    return tables


### Retrieval multi-relevance + matching/ranking audit

MRR/Recall/NDCG benar untuk banyak dokumen relevan. Corpus index bukan supervised train. Query validation/test dipisahkan menurut exact query.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer


def prepare_retrieval(corpus, queries, cfg):
    c = corpus.rename(columns={cfg['doc_id_col']:'doc_id',cfg['doc_text_col']:'text'}).copy()
    q = queries.rename(columns={cfg['query_col']:'query',cfg['relevant_col']:'relevant_ids'}).copy()
    assert {'doc_id','text'}.issubset(c) and {'query','relevant_ids'}.issubset(q)
    assert c.doc_id.notna().all() and c.doc_id.astype(str).is_unique, 'doc_id harus unik dan terisi'
    c['doc_id'] = c.doc_id.astype(str)
    c['text'] = c.text.map(safe_text)
    assert c.text.ne('').all(), 'Dokumen kosong perlu ditangani dahulu'
    q['query'] = q['query'].map(safe_text)
    assert q['query'].ne('').all(), 'Query kosong tidak dapat dievaluasi'
    def parse(value):
        value = json.loads(value) if isinstance(value,str) else value
        assert isinstance(value,list), 'relevant_ids harus JSON array, contoh ["d1","d2"]'
        return set(map(str,value))
    q['gold'] = q.relevant_ids.map(parse)
    assert q.gold.map(bool).all(), 'Query tanpa judgment positif belum dapat dinilai'
    assert set().union(*q.gold).issubset(set(c.doc_id)), 'Gold doc tidak ada dalam corpus'
    q['_group'] = pd.factorize(q['query'].str.casefold())[0]
    val,test = next(GroupShuffleSplit(1,test_size=.5,random_state=cfg['seed']).split(q,groups=q._group))
    print('Corpus:',len(c),'| query validation/test:',len(val),len(test))
    print('Duplikat dokumen:',c.text.str.casefold().duplicated().sum(),'| judgment per query:',q.gold.map(len).describe())
    return {'corpus':c.reset_index(drop=True),'val':q.iloc[val].reset_index(drop=True),'test':q.iloc[test].reset_index(drop=True)}


def retrieval_metrics(ranking, gold, doc_ids):
    rows = []
    ids = np.asarray(doc_ids).astype(str)
    for order,relevant in zip(ranking,gold):
        retrieved = ids[order].tolist()
        hits = np.array([d in relevant for d in retrieved],dtype=float)
        positions = np.flatnonzero(hits[:10])
        ideal = np.sum(1/np.log2(np.arange(min(len(relevant),10))+2))
        rows.append({'MRR@10':1/(positions[0]+1) if len(positions) else 0,
            **{f'Recall@{k}':len(set(retrieved[:k])&relevant)/len(relevant) for k in [1,5,10]},
            'NDCG@10':float(np.sum(hits[:10]/np.log2(np.arange(len(hits[:10]))+2))/ideal)})
    return pd.DataFrame(rows).mean().to_dict()


def lexical_rankings(corpus, queries, method='bm25', top=100):
    if method=='bm25':
        from rank_bm25 import BM25Okapi
        tokenize = lambda t:re.findall(r'[^\W_]+',t.casefold())
        index = BM25Okapi([tokenize(t) for t in corpus])
        return np.array([np.argsort(-index.get_scores(tokenize(t)),kind='stable')[:top] for t in queries])
    vec = TfidfVectorizer(ngram_range=(1,2),min_df=1,max_features=80000).fit(corpus)
    docs = vec.transform(corpus)
    return np.array([np.argsort(-(vec.transform([q])@docs.T).toarray()[0],kind='stable')[:top] for q in queries])


def rrf_rankings(first, second, alpha=.5, top=100, constant=60):
    merged = []
    for a,b in zip(first,second):
        scores = {}
        for ranking,weight in [(a,1-alpha),(b,alpha)]:
            for rank,doc in enumerate(ranking):
                scores[int(doc)] = scores.get(int(doc),0)+weight/(constant+rank+1)
        merged.append(sorted(scores,key=lambda doc:(-scores[doc],doc))[:top])
    return np.asarray(merged)


def dense_rankings(corpus, queries, cfg, top=100):
    chunks,owners = [],[]
    for i,text in enumerate(corpus):
        words = text.split()
        for start in range(0,min(len(words),cfg.get('max_chunks',6)*120),120):
            chunks.append(' '.join(words[start:start+120]));owners.append(i)
    emb = encode_e5(chunks,cfg,'passage')
    q = encode_e5(queries,cfg,'query')
    rankings = []
    for vector in q:
        scores = np.full(len(corpus),-np.inf)
        np.maximum.at(scores,owners,emb@vector)
        rankings.append(np.argsort(-scores,kind='stable')[:top])
    return np.array(rankings)


def rerank_candidates(corpus, queries, rankings, cfg, top=20):
    require_kaggle(gpu=True)
    import gc
    import torch
    from transformers import AutoTokenizer,AutoModelForSequenceClassification
    repo = cfg.get('reranker','cross-encoder/mmarco-mMiniLMv2-L12-H384-v1')
    tok = AutoTokenizer.from_pretrained(repo,local_files_only=cfg.get('offline',False))
    model = AutoModelForSequenceClassification.from_pretrained(repo,local_files_only=cfg.get('offline',False)).cuda().eval()
    assert model.config.num_labels==1, 'Reranker ini mengharapkan satu skor relevansi'
    result = []
    with torch.inference_mode():
        for query,ranking in zip(queries,rankings):
            candidates = ranking[:top]
            enc = tok([query]*len(candidates),[corpus[int(i)] for i in candidates],padding=True,truncation=True,max_length=cfg['max_len'],return_tensors='pt').to('cuda')
            scores = model(**enc).logits.float().cpu().numpy().ravel()
            result.append(np.r_[candidates[np.argsort(-scores,kind='stable')],ranking[top:]])
    del model
    gc.collect();torch.cuda.empty_cache()
    return np.array(result)


def run_retrieval(data, cfg):
    c = data['corpus'];doc_ids=c.doc_id.to_numpy();top=min(100,len(c))
    rankings = {split:{} for split in ['val','test']}
    for split in rankings:
        q = data[split]['query'].tolist()
        for method in [m for m in cfg['models'] if m in {'tfidf','bm25'}]:
            rankings[split][method] = lexical_rankings(c.text.tolist(),q,method,top)
        if 'e5' in cfg['models']:
            rankings[split]['e5'] = dense_rankings(c.text.tolist(),q,cfg,top)
    if 'rrf' in cfg['models']:
        assert all(name in rankings['val'] for name in ['bm25','e5']), 'RRF memerlukan BM25 dan E5'
        alpha = max([0,.25,.5,.75,1],key=lambda a:retrieval_metrics(rrf_rankings(rankings['val']['bm25'],rankings['val']['e5'],a,top),data['val'].gold,doc_ids)['MRR@10'])
        for split in rankings:
            rankings[split]['rrf'] = rrf_rankings(rankings[split]['bm25'],rankings[split]['e5'],alpha,top)
    if 'reranker' in cfg['models']:
        base = 'rrf' if 'rrf' in rankings['val'] else 'bm25'
        assert base in rankings['val'], 'Reranker memerlukan kandidat BM25 atau RRF'
        for split in rankings:
            rankings[split]['reranker'] = rerank_candidates(c.text.tolist(),data[split]['query'].tolist(),rankings[split][base],cfg)
    validation = pd.DataFrame([{'model':name,**retrieval_metrics(order,data['val'].gold,doc_ids)} for name,order in rankings['val'].items()])
    assert len(validation), 'Pilih minimal satu model retrieval'
    chosen = validation.sort_values('MRR@10',ascending=False,kind='stable').iloc[0].model
    out = Path(cfg['out']);out.mkdir(parents=True,exist_ok=True)
    (out/'selection_frozen_before_test.json').write_text(json.dumps({'model':chosen,'rrf_alpha':alpha if 'rrf' in cfg['models'] else None}))
    holdout = pd.DataFrame([{'model':name,**retrieval_metrics(order,data['test'].gold,doc_ids)} for name,order in rankings['test'].items()])
    rows = data['test'][['query']].copy()
    rows['gold'] = data['test'].gold.map(lambda v:sorted(v))
    rows['top10'] = [[doc_ids[i] for i in r[:10]] for r in rankings['test'][chosen]]
    rows['hit_at_1'] = [str(doc_ids[r[0]]) in g for r,g in zip(rankings['test'][chosen],data['test'].gold)]
    tables = {'validation':validation,'holdout':holdout,'errors_and_rankings':rows}
    save_tables(tables,out)
    for name,table in tables.items():
        show_table(name,table.head(20))
    print('FINAL (validation):',chosen,'| Judgment yang tidak lengkap dapat menurunkan Recall/NDCG secara semu.')
    return tables


### E5 masked mean pooling + prefix



In [ ]:
def e5_encode(texts, cfg, role='query'):
    """E5: masked mean pooling, L2 normalisasi, prefix query/passage yang eksplisit."""
    require_kaggle(gpu=True)
    import gc
    import torch
    import torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModel
    assert role in {'query','passage'}
    repo = cfg.get('e5_model','intfloat/multilingual-e5-base')
    tok = AutoTokenizer.from_pretrained(repo,local_files_only=cfg.get('offline',False))
    model = AutoModel.from_pretrained(repo,local_files_only=cfg.get('offline',False)).cuda().eval()
    rows = []
    texts = list(texts)
    with torch.inference_mode():
        for start in range(0,len(texts),cfg.get('embed_batch',32)):
            batch = [role+': '+str(t) for t in texts[start:start+cfg.get('embed_batch',32)]]
            enc = tok(batch,padding=True,truncation=True,max_length=cfg.get('max_len',256),return_tensors='pt').to('cuda')
            hidden = model(**enc).last_hidden_state.float()
            pooled = (hidden*enc.attention_mask.unsqueeze(-1)).sum(1)/enc.attention_mask.sum(1,keepdim=True).clamp(min=1)
            rows.append(F.normalize(pooled,p=2,dim=1).cpu().numpy())
    result = np.vstack(rows) if rows else np.empty((0,model.config.hidden_size),dtype='float32')
    del model
    gc.collect();torch.cuda.empty_cache()
    return result


ENCODERS['e5'] = e5_encode


## 4 · Jalankan

Holdout dibuka setelah pemilihan validation dibekukan. Selisih kecil perlu repeated group split. EDA angka tersedia sebelum modeling.

In [ ]:
require_kaggle(gpu=True)
seed_all(CFG['seed'])

### Corpus & queries

In [ ]:
corpus = read_table(CFG['corpus_file'], CFG['encoding'], string_cols=[CFG['doc_id_col']])
queries = read_table(CFG['queries_file'], CFG['encoding'])
tables = eda_for_task(corpus, CFG)
data = prepare_retrieval(corpus, queries, CFG)

### Ranking → validation → holdout

In [ ]:
result = run_retrieval(data, CFG)